# Day 4 Instructor Demo: the production version (optional)

**Run by the instructor only, locally, against the Docker stack** (`make up`: gateway, postgres, redis, api, all
upstream of mock-llm by default, so it rehearses with no provider key). The guide's talk sections are slides
only; this notebook shows what participants did in library mode, done by the LiteLLM **proxy**: virtual keys,
budgets and rate limits enforced on every call, shared by every app. Repeatable: run it top to bottom as often as
you like (keys are recreated by alias). TinyCoder is fictional.

In [ ]:
# Setup: talks to the Docker stack (`make up`): the LiteLLM proxy, Postgres, Redis and the api in proxy mode.
# Inside the compose network (make exec-notebooks) the services are gateway:4000 and api:8000; from the host,
# localhost:4000 and localhost:8200. Keys come from the environment and keys.local.json; none is printed in full.
import json, os, subprocess, sys, time
from pathlib import Path
import httpx

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents, Path("/workspace")] if (p / "gateway" / "litellm_config.yaml").exists())
sys.path.insert(0, str(ROOT))
os.environ.setdefault("LITELLM_LOCAL_MODEL_COST_MAP", "True")
if not os.environ.get("LITELLM_MASTER_KEY") and (ROOT / ".env").exists():
    for line in (ROOT / ".env").read_text().splitlines():
        if line.startswith("LITELLM_MASTER_KEY="):
            os.environ["LITELLM_MASTER_KEY"] = line.split("=", 1)[1].split("#")[0].strip()

def reachable(url):
    try:
        return httpx.get(url, timeout=3, trust_env=False).status_code < 500
    except httpx.HTTPError:
        return False

GATEWAY = os.environ.get("GATEWAY_URL") or ("http://gateway:4000" if reachable("http://gateway:4000/health/liveliness") else "http://localhost:4000")
API = os.environ.get("API_URL") or ("http://api:8000" if reachable("http://api:8000/health") else "http://localhost:8200")
os.environ["GATEWAY_URL"] = GATEWAY
from paper_agent.fixtures.tinycoder_pdf import make_pdf
from paper_agent.service import keys
from paper_agent.service.keys import mask
MASTER = {"Authorization": f"Bearer {os.environ['LITELLM_MASTER_KEY']}"}
gw = httpx.Client(base_url=GATEWAY, timeout=60, trust_env=False)
api = httpx.Client(base_url=API, timeout=120, trust_env=False)
Q_SPEED = "How much faster is TinyCoder than the 7B baseline?"
print("gateway:", GATEWAY, "| api:", API, "| master key:", mask(os.environ["LITELLM_MASTER_KEY"]))

gateway: http://gateway:4000 | api: http://api:8000 | master key: sk-li…e-me


---
## D0 · Before class: is everything up?

In [2]:
# D0
ready = gw.get("/health/readiness", headers=MASTER).json()
cache = gw.get("/cache/ping", headers=MASTER).json()
print("gateway  :", gw.get("/health/liveliness").json(), "|", ready.get("status"))
print("postgres :", ready.get("db"), "(virtual keys, budgets, spend)")
print("redis    :", cache.get("cache_type"), "ping", cache.get("ping_response"), "(rate-limit counters, response cache)")
print("api      :", api.get("/health").json())
print("aliases  :", sorted(m["id"] for m in gw.get("/v1/models", headers=MASTER).json()["data"]))

gateway  : I'm alive! | healthy
postgres : connected (virtual keys, budgets, spend)
redis    : redis ping True (rate-limit counters, response cache)
api      : {'status': 'ok', 'gateway_mode': 'proxy'}
aliases  : ['paper-embed', 'paper-explainer', 'paper-explainer-broken', 'paper-explainer-fallback']


---
## D5.1 · The same gateway, as config · *slide: The gateway: Day 1, now as config*

Left: the Router participants print in 5.1. Right: `gateway/litellm_config.yaml`, which the proxy runs.

In [3]:
# D5.1
import yaml
cfg = yaml.safe_load((ROOT / "gateway" / "litellm_config.yaml").read_text())
os.environ.setdefault("LLM_BASE_URL", "http://mock-llm:8000/v1"); os.environ.setdefault("LLM_API_KEY", "sk-mock-local")
os.environ.setdefault("LLM_MODEL", "mock-llm"); os.environ.setdefault("LLM_FALLBACK_MODEL", "mock-llm-fallback")
from paper_agent.service.gateway import make_router
router = make_router()
rs = cfg["router_settings"]
rows = [("alias -> model", ", ".join(f"{d['model_name']}->{d['model_info']['id']}" for d in router.model_list if d["model_info"]["id"] != "embed"),
         ", ".join(f"{m['model_name']}->{m['model_info']['id']}" for m in cfg["model_list"] if m["model_info"]["id"] not in ("embed", "broken-primary"))),
        ("fallbacks", json.dumps(router.fallbacks), json.dumps([f for f in rs["fallbacks"] if "paper-explainer" in f])),
        ("num_retries", router.num_retries, rs["num_retries"]),
        ("timeout (s)", router.timeout, rs["timeout"]),
        ("max_tokens", "400 (make_chat_model)", cfg["model_list"][0]["litellm_params"]["max_tokens"]),
        ("budgets", "our @before_model guard", "virtual keys (Postgres)"),
        ("rate limits", "in-process counter", "rpm_limit per key (Redis)")]
print(f"{'':16} {'notebook: litellm.Router (library)':58} {'production: litellm_config.yaml (proxy)'}")
for name, lib, proxy in rows:
    print(f"{name:16} {str(lib):58} {proxy}")

                 notebook: litellm.Router (library)                         production: litellm_config.yaml (proxy)
alias -> model   paper-explainer->primary, paper-explainer-fallback->fallback paper-explainer->primary, paper-explainer-fallback->fallback
fallbacks        [{"paper-explainer": ["paper-explainer-fallback"]}]        [{"paper-explainer": ["paper-explainer-fallback"]}]
num_retries      2                                                          2
timeout (s)      30                                                         30
max_tokens       400 (make_chat_model)                                      400
budgets          our @before_model guard                                    virtual keys (Postgres)
rate limits      in-process counter                                         rpm_limit per key (Redis)


---
## D5.2 · Fallback through the proxy

`paper-explainer-broken` points at a model the provider doesn't have; the proxy falls back. The response header says which deployment answered.

In [4]:
# D5.2
r = gw.post("/v1/chat/completions", headers=MASTER, json={"model": "paper-explainer-broken", "max_tokens": 20,
            "messages": [{"role": "user", "content": "Reply with one word: ready"}]})
print("status:", r.status_code)
for h in ("x-litellm-model-id", "x-litellm-model-group", "x-litellm-attempted-fallbacks", "x-litellm-attempted-retries"):
    print(f"{h:32} {r.headers.get(h)}")

status: 200
x-litellm-model-id               fallback
x-litellm-model-group            paper-explainer-fallback
x-litellm-attempted-fallbacks    1
x-litellm-attempted-retries      0


---
## D5.6a · One virtual key per user · *slide: Limits and spend*

`POST /key/generate`. Alice gets $1.00 a day and 10 requests a minute. Bob's budget is about 1.5 questions, measured from one real question.

In [5]:
# D5.6a
ALICE_SPEC = {"user_id": "alice", "max_budget": 1.00, "budget_duration": "1d", "rpm_limit": 10}
keys.delete_keys(["alice", "bob"])
print("POST /key/generate", json.dumps({"key_alias": "alice", "models": keys.MODELS, **ALICE_SPEC}))
ALICE_KEY = keys.generate_key("alice", **ALICE_SPEC)["key"]
print("  → key", mask(ALICE_KEY))
PAPER_ID = api.post("/papers", headers={"Authorization": f"Bearer {ALICE_KEY}"},
                    files={"file": ("tinycoder.pdf", make_pdf(), "application/pdf")}).json()["paper_id"]
one = api.post(f"/papers/{PAPER_ID}/ask", headers={"Authorization": f"Bearer {ALICE_KEY}"}, json={"question": Q_SPEED}).json()
BOB_SPEC = {"user_id": "bob", "max_budget": round(1.5 * one["cost_usd"], 6), "budget_duration": "1d", "rpm_limit": 10}
print(f"\none question costs ${one['cost_usd']:.6f} → bob's budget ${BOB_SPEC['max_budget']:.6f}")
print("POST /key/generate", json.dumps({"key_alias": "bob", "models": keys.MODELS, **BOB_SPEC}))
BOB_KEY = keys.generate_key("bob", **BOB_SPEC)["key"]
print("  → key", mask(BOB_KEY))
_ = Path(keys.KEYS_FILE).write_text(json.dumps({**keys.load_keys(), "alice": ALICE_KEY, "bob": BOB_KEY}, indent=2))

POST /key/generate {"key_alias": "alice", "models": ["paper-explainer", "paper-explainer-fallback", "paper-embed"], "user_id": "alice", "max_budget": 1.0, "budget_duration": "1d", "rpm_limit": 10}
  → key sk-I6…_OFA

one question costs $0.000339 → bob's budget $0.000509
POST /key/generate {"key_alias": "bob", "models": ["paper-explainer", "paper-explainer-fallback", "paper-embed"], "user_id": "bob", "max_budget": 0.000509, "budget_duration": "1d", "rpm_limit": 10}
  → key sk-qH…D2tQ


---
## D5.6b · Bob runs out, through the api

The proxy refuses Bob's next model call; our api turns that into the same friendly 429 participants saw in 5.4. First the proxy's raw error, once.

In [6]:
# D5.6b
BOB_H = {"Authorization": f"Bearer {BOB_KEY}"}
for n in (1, 2, 3):
    r = api.post(f"/papers/{PAPER_ID}/ask", headers=BOB_H, json={"question": Q_SPEED})
    print(f"bob, question {n}: {r.status_code}", "" if r.status_code == 200 else r.json()["error"])
raw = gw.post("/v1/chat/completions", headers=BOB_H, json={"model": "paper-explainer", "max_tokens": 5,
              "messages": [{"role": "user", "content": "hi"}]})
err = raw.json()["error"]
print(f"\nthe proxy, raw : HTTP {raw.status_code} type={err['type']!r}\n                 {err['message'].split('(sk-')[0]}…")
print("our api        :", r.status_code, json.dumps(r.json()))

bob, question 1: 200 
bob, question 2: 200 
bob, question 3: 429 budget_exceeded

the proxy, raw : HTTP 422 type='budget_exceeded'
                 Budget has been exceeded! Key=bob …
our api        : 429 {"error": "budget_exceeded", "message": "You've used your budget for today ($0.000677 of $0.000509). It resets at midnight UTC; ask your admin if you need more."}


---
## D5.6c · Rate limit through the proxy

Every model and embedding call counts against Alice's 10 requests per minute (one question is about three).

In [7]:
# D5.6c
ALICE_H = {"Authorization": f"Bearer {ALICE_KEY}"}
for n in range(1, 8):
    r = api.post(f"/papers/{PAPER_ID}/ask", headers=ALICE_H, json={"question": Q_SPEED})
    print(f"alice, question {n}: {r.status_code}", "" if r.status_code == 200 else
          f"Retry-After: {r.headers.get('Retry-After')} · {r.json()['message']}")
    if r.status_code == 429:
        break

alice, question 1: 200 
alice, question 2: 200 
alice, question 3: 200 
alice, question 4: 429 Retry-After: 60 · Too many requests. Try again in 60 seconds.


---
## D5.6d · Spend per key

`GET /key/info` next to the api's `/me/usage`, which reads it in proxy mode. The proxy writes spend to Postgres in batches: the first read can lag a few seconds.

In [8]:
# D5.6d
deadline = time.monotonic() + 90
while time.monotonic() < deadline and not all(keys.key_info(k)["spend"] for k in (ALICE_KEY, BOB_KEY)):
    time.sleep(3)
for user, key in (("alice", ALICE_KEY), ("bob", BOB_KEY)):
    info = keys.key_info(key)
    me = api.get("/me/usage", headers={"Authorization": f"Bearer {key}"}).json()
    print(f"{user:6} /key/info spend ${info['spend']:.6f} of ${info['max_budget']} (resets {info.get('budget_reset_at', '?')[:10]})"
          f"  |  /me/usage {json.dumps(me)}")

alice  /key/info spend $0.001444 of $1.0 (resets 2026-10-10)  |  /me/usage {"user": "alice", "llm_calls": null, "spent_usd": 0.001444, "budget_usd": 1.0, "remaining_usd": 0.998556, "source": "gateway"}
bob    /key/info spend $0.000677 of $0.000509 (resets 2026-10-10)  |  /me/usage {"user": "bob", "llm_calls": null, "spent_usd": 0.000677, "budget_usd": 0.000509, "remaining_usd": -0.000168, "source": "gateway"}


---
## D6 · The same spans in production

The api's code, in proxy mode, with the console exporter (the api service's default: `docker compose logs api`). Same span names as 6.2.

In [9]:
# D6
from fastapi.testclient import TestClient
from paper_agent.service import config
from paper_agent.service.app import build_app
from paper_agent.service.pipeline import build_agent
from paper_agent.service.store import PaperStore, setup_retrieval
from paper_agent.service.tracing import one_line, setup_tracing
from opentelemetry.sdk.trace.export import ConsoleSpanExporter
config.GATEWAY_MODE, config.GATEWAY_URL = "proxy", GATEWAY
setup_retrieval(None, "/tmp/d6")
setup_tracing(ConsoleSpanExporter(formatter=one_line))
CAROL_H = {"Authorization": f"Bearer {keys.recreate_keys({'carol': {'user_id': 'carol', 'max_budget': 1.0}})['carol']}"}
with TestClient(build_app(build_agent(), PaperStore("/tmp/d6"))) as c:
    pid = c.post("/papers", headers=CAROL_H, files={"file": ("tinycoder.pdf", make_pdf(), "application/pdf")}).json()["paper_id"]
    print("---- one question ----")
    r = c.post(f"/papers/{pid}/ask", headers=CAROL_H, json={"question": "What attention mechanism does TinyCoder use?"})
setup_tracing(ConsoleSpanExporter(formatter=lambda s: ""))       # quiet again
print("status", r.status_code, "| answered_by", r.json().get("answered_by"))

{"span": "fastapi.dependencies", "trace_id": "79e5fd17", "ms": 17.0, "code.function.name": "paper_agent.service.app.build_app.<locals>.upload"}
{"span": "parse_pdf", "trace_id": "79e5fd17", "ms": 107.1, "pdf.chunks": 6}
embedded 6 chunks with mock-embed (vector length 1024); cached as c03739855bbdcf3c-mock-embed-ea3dbc19.json
{"span": "embed_chunks", "trace_id": "79e5fd17", "ms": 109.5, "embed.model": "mock-embed"}
{"span": "fastapi.endpoint", "trace_id": "79e5fd17", "ms": 218.2, "code.function.name": "paper_agent.service.app.build_app.<locals>.upload"}
{"span": "fastapi.serialization", "trace_id": "79e5fd17", "ms": 0.1, "code.function.name": "paper_agent.service.app.build_app.<locals>.upload"}
{"span": "POST /papers", "trace_id": "79e5fd17", "ms": 239.4}
---- one question ----
{"span": "fastapi.dependencies", "trace_id": "c1830bed", "ms": 0.3, "code.function.name": "paper_agent.service.app.build_app.<locals>.ask"}
{"span": "llm.call", "trace_id": "c1830bed", "ms": 87.5, "llm.deploymen

---
## D7 · The gate in CI · *slide: The gate*

`make eval`: the 20-question set through the api, metrics against `paper_agent/evals/baseline.json`, and an exit code a CI job can fail on. Then the v2 prompt.

In [10]:
# D7
EVAL_KEY = keys.recreate_keys({"eval-bot": keys.DEMO_KEYS["eval-bot"]})["eval-bot"]
for version in ("v1", "v2"):
    print(f"$ make eval PROMPT_VERSION={version}")
    run = subprocess.run([sys.executable, "-m", "paper_agent.evals.runner", "--api", API, "--gateway", GATEWAY,
                          "--prompt-version", version], cwd=ROOT, capture_output=True, text=True,
                         env={**os.environ, "EVAL_KEY": EVAL_KEY, "PYTHONPATH": str(ROOT)})
    print(run.stdout or run.stderr[-1500:])
    print(f"exit code: {run.returncode}  ({'the build passes' if run.returncode == 0 else 'CI fails the build'})\n")

$ make eval PROMPT_VERSION=v1
golden set: 20 questions · prompt v1 · judge judge_v1 · paper c03739855bbd via http://api:8000

✅ What is TinyCoder's pass@1 compared with the 7B baseline?      
✅ What attention mechanism does TinyCoder use?                   
❌ What are the main limitations of TinyCoder?                    failed: right_section
✅ How much faster is TinyCoder than the 7B baseline?             
✅ How does TinyCoder do on files longer than 1,024 tokens?       
✅ What data was TinyCoder trained on?                            
✅ Who funded the TinyCoder project?                              
✅ What learning rate was used to train TinyCoder?                
✅ How many layers does TinyCoder have?                           
✅ How big is TinyCoder's attention window?                       
✅ How many GPUs were used to train TinyCoder, and for how long?  
❌ How was TinyCoder evaluated?                                   failed: right_section, page_accuracy
✅ What does pass@1 mean i

---
## DX · After class: clean up

Delete the demo keys, so the next rehearsal starts clean (`POST /key/delete` by alias).

In [ ]:
# DX
print("deleted:", keys.delete_keys(["alice", "bob", "carol", "eval-bot"]), "key(s)")
r = httpx.get(f"{GATEWAY}/key/info", params={"key": BOB_KEY}, headers={"Authorization": f"Bearer {BOB_KEY}"}, trust_env=False)
print("bob's old key now:", r.status_code, r.json()["error"]["type"] if r.status_code != 200 else "still works?!")